In [ ]:
!pip install transformers jiwer librosa soundfile torch accelerate

In [ ]:
import os
import torch
from transformers import pipeline
from jiwer import wer

# 1. Khởi tạo phần cứng và mô hình
device = "cuda:0" if torch.cuda.is_available() else "cpu"
print(f"Đang sử dụng phần cứng: {device}")

# Sử dụng Whisper bản 'small' để cân bằng giữa tốc độ và độ chính xác
pipe = pipeline(
    "automatic-speech-recognition",
    model="vinai/PhoWhisper-small",
    device=device,
    chunk_length_s=30
)
# 2. Cấu hình đường dẫn dữ liệu 
base_path = "/kaggle/input/datasets/tuannguyenvananh/vivos-dataset/vivos" 
test_path = os.path.join(base_path, "test")
prompts_file = os.path.join(test_path, "prompts.txt")
waves_dir = os.path.join(test_path, "waves")

# 3. Đọc dữ liệu chuẩn 
ground_truths = {}
with open(prompts_file, 'r', encoding='utf-8') as f:
    for line in f:
        # File có định dạng: VIVOSDEV01_001 nội dung văn bản
        parts = line.strip().split(' ', 1)
        if len(parts) == 2:
            audio_id, transcript = parts
            ground_truths[audio_id] = transcript.lower()

import librosa # Thêm thư viện đọc audio

# 4. Quá trình kiểm thử 
predictions = []
references = []
count = 0
max_test = 20 

total_duration = 0.0 # Biến lưu tổng thời lượng của tất cả audio đã xử lý

import re

# Thêm hàm xóa dấu câu
def clean_text(text):
    text = text.lower().strip()
    text = re.sub(r'[^\w\s]', '', text) # Xóa các ký tự không phải chữ/số/khoảng trắng
    return text

print("Bắt đầu nhận diện...")
for speaker in os.listdir(waves_dir):
    speaker_dir = os.path.join(waves_dir, speaker)
    if not os.path.isdir(speaker_dir):
        continue

    for audio_file in os.listdir(speaker_dir):
        if audio_file.endswith(".wav"):
            audio_id = audio_file.replace(".wav", "")
            
            if audio_id not in ground_truths:
                continue
                
            audio_path = os.path.join(speaker_dir, audio_file)
            
            # Đọc file bằng librosa và đưa về tần số lấy mẫu 16kHz (bắt buộc với Whisper)
            speech, sample_rate = librosa.load(audio_path, sr=16000)
            
            # TÍNH THỜI LƯỢNG AUDIO: Số lượng sample / Tần số lấy mẫu
            file_duration = len(speech) / sample_rate
            total_duration += file_duration
            
            result = pipe(speech, generate_kwargs={"language": "vietnamese"})
            pred_text = clean_text(result["text"])
            
            predictions.append(pred_text)
            references.append(ground_truths[audio_id])
            
            print(f"[{count+1}] Audio ID: {audio_id} (Thời lượng: {file_duration:.2f}s)")
            print(f"Chuẩn: {ground_truths[audio_id]}")
            print(f"AI   : {pred_text}\n")
            
            count += 1
            if max_test and count >= max_test:
                break
    if max_test and count >= max_test:
        break

# 5. Đánh giá % lỗi (WER) và Thống kê thời gian
if len(predictions) > 0:
    error_rate = wer(references, predictions)
    avg_duration = total_duration / count
    
    print(f"==> Tỷ lệ lỗi (WER) trên {count} mẫu: {error_rate * 100:.2f}%")
    print(f"==> Thời gian trung bình mỗi file: {avg_duration:.2f} giây")
else:
    print("Lỗi: Không tìm thấy file dữ liệu nào hợp lệ!")

In [7]:
!pip install transformers torch librosa accelerate


In [9]:
import os
import torch
import librosa
from transformers import pipeline, AutoTokenizer, AutoModelForCausalLM

# ==========================================
# 1. KHỞI TẠO PHẦN CỨNG
# ==========================================
device = "cuda:0" if torch.cuda.is_available() else "cpu"
print(f"🚀 Đang sử dụng phần cứng: {device}")

# ==========================================
# 2. TẢI MÔ HÌNH NHẬN DIỆN GIỌNG NÓI (PhoWhisper)
# ==========================================
print("⏳ Đang tải mô hình PhoWhisper (STT)...")
stt_pipe = pipeline(
    "automatic-speech-recognition",
    model="vinai/PhoWhisper-small",
    device=device,
    chunk_length_s=30 
)

# ==========================================
# 3. TẢI MÔ HÌNH TÓM TẮT (Qwen2.5 1.5B)
# ==========================================
print("⏳ Đang tải mô hình Qwen2.5-1.5B-Instruct (Summarization)...")
# Dùng float16 để giảm một nửa lượng RAM tiêu thụ, chạy cực mượt trên Kaggle
llm_model_id = "Qwen/Qwen2.5-1.5B-Instruct"
llm_tokenizer = AutoTokenizer.from_pretrained(llm_model_id)
llm_model = AutoModelForCausalLM.from_pretrained(
    llm_model_id, 
    torch_dtype=torch.float16, 
    device_map=device
)
print("✅ Tải các mô hình hoàn tất!\n")

# ==========================================
# 4. ĐỌC DỮ LIỆU & NHẬN DIỆN GIỌNG NÓI (5 PHÚT ĐẦU)
# ==========================================
audio_path = "/kaggle/input/datasets/lucking47/testing-mp3/TRC TIP- Khai mc K hp khng thng l th Nht Quc hi kha XVI - VOV.mp3"

print("🎙️ Bắt đầu đọc 30 phút đầu của file audio...")
speech, sample_rate = librosa.load(audio_path, sr=16000, duration=1800)

stt_result = stt_pipe(speech, generate_kwargs={"language": "vietnamese"})
full_transcript = stt_result["text"]

print("\n📝 VĂN BẢN GỐC (Trích 500 ký tự đầu):")
print(full_transcript[:500] + "...\n")

# ==========================================
# 5. TÓM TẮT TRỰC TIẾP VỚI LLM
# ==========================================
print("💡 Đang nhờ AI tóm tắt văn bản (Vui lòng đợi vài chục giây)...\n")

# Tạo cấu trúc giao tiếp với AI
messages = [
    {"role": "system", "content": "Bạn là một trợ lý AI chuyên nghiệp. Nhiệm vụ của bạn là đọc bản ghi âm cuộc họp và tóm tắt những ý chính quan trọng nhất dưới dạng các gạch đầu dòng ngắn gọn, dễ hiểu."},
    {"role": "user", "content": f"Hãy tóm tắt nội dung sau:\n\n{full_transcript}"}
]

# Chuẩn bị dữ liệu đầu vào cho LLM
text = llm_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = llm_tokenizer([text], return_tensors="pt").to(device)

# Tiến hành sinh tóm tắt
with torch.no_grad():
    outputs = llm_model.generate(
        **inputs,
        max_new_tokens=400, # Độ dài tối đa của bản tóm tắt
        temperature=0.3,    # Giữ nhiệt độ thấp để AI nói đúng sự thật, không bịa đặt
        do_sample=True
    )

# Lấy phần text sinh ra mới nhất (loại bỏ phần prompt truyền vào)
generated_ids = outputs[0][inputs.input_ids.shape[1]:]
summary_text = llm_tokenizer.decode(generated_ids, skip_special_tokens=True)

# ==========================================
# 6. IN KẾT QUẢ VÀ LƯU FILE
# ==========================================
print("\n" + "="*50)
print("🏆 BẢN TÓM TẮT (5 PHÚT ĐẦU)")
print("="*50)
print(summary_text)

# Lưu kết quả
with open("/kaggle/working/ket_qua_tom_tat_5p_qwen.txt", "w", encoding="utf-8") as f:
    f.write("VĂN BẢN GỐC (5 PHÚT ĐẦU):\n" + full_transcript + "\n\n")
    f.write("BẢN TÓM TẮT:\n" + summary_text)
print("\n💾 Đã lưu kết quả vào thư mục /kaggle/working/ket_qua_tom_tat_5p_qwen.txt")

🚀 Đang sử dụng phần cứng: cuda:0
⏳ Đang tải mô hình PhoWhisper (STT)...


Exception in thread Thread-auto_conversion:
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_http.py", line 761, in hf_raise_for_status
    response.raise_for_status()
  File "/usr/local/lib/python3.12/dist-packages/httpx/_models.py", line 829, in raise_for_status
    raise HTTPStatusError(message, request=request, response=self)
httpx.HTTPStatusError: Client error '403 Forbidden' for url 'https://huggingface.co/api/models/vinai/PhoWhisper-small/discussions?p=0'
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/403

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "/usr/lib/python3.12/threading.py", line 1075, in _bootstrap_inner
    self.run()
  File "/usr/lib/python3.12/threading.py", line 1012, in run
    self._target(*self._args, **self._kwargs)
  File "/usr/local/lib/python3.12/dist-packages/transformers/safetensors_conversion.p

Loading weights:   0%|          | 0/480 [00:00<?, ?it/s]

    hf_raise_for_status(resp)
  File "/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_http.py", line 849, in hf_raise_for_status
    raise _format(HfHubHTTPError, message, response) from e
huggingface_hub.errors.HfHubHTTPError: (Request ID: Root=1-6aa949ec-3df192e37d2c978b63f2510a;2b563ef4-7f71-4d34-8e6d-cff21b63607b)

403 Forbidden: Discussions are disabled for this repo.
Cannot access content at: https://huggingface.co/api/models/vinai/PhoWhisper-small/discussions?p=0.
Make sure your token has the correct permissions.
The tied weights mapping and config for this model specifies to tie model.decoder.embed_tokens.weight to proj_out.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
Using `chunk_length_s` is very experimental with seq2seq models. The results will not necessarily be entirely accurate and will have caveats. More information: https://github.com/huggin

⏳ Đang tải mô hình Qwen2.5-1.5B-Instruct (Summarization)...


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

✅ Tải các mô hình hoàn tất!

🎙️ Bắt đầu đọc 30 phút đầu của file audio...

📝 VĂN BẢN GỐC (Trích 500 ký tự đầu):
trân trọng kính mời các đồng chí lãnh đạo nguyên lãnh đạo đảng nhà nước mặt trận tổ quốc việt nam các vị đtrước những lời cao cả đường phước chiến thắng vắng đời chưa xuống ngoài da chè tre vẫn hạnh phúc hạnh phúc hạnh phúc hạnh phúc hạnh phúc bay cho nguồn cầu thời gian hoang mang cho đến chiều chuông vị đêm khắp trên giông góc khuôn nguyễn bảo tơ hoa chương tiên truyền thương tiên truyền thương tư truyền thương tư tư tư tư tư tư tư tư tư tư tư vị đại biểu cùng toàn thể các đồng chí tô lâm tổng...

💡 Đang nhờ AI tóm tắt văn bản (Vui lòng đợi vài chục giây)...


🏆 BẢN TÓM TẮT (5 PHÚT ĐẦU)
Tóm tắt nội dung cuộc họp:

- Quốc hội khóa 16 khai mạc kỳ họp không thường lệ thứ nhất.
- Chủ đề: Xem xét quyết định về ba nhóm nội dung: 1) Xem xét thông qua mười năm dự án luật; 2) Xem xét quyết định một số vấn đề quan trọng của đất nước; 3) Xem xét quyết định về công tác nhân sự.
- Các v